In [ ]:
from google.colab import files
uploaded = files.upload()

Saving combined_lora_adapter.zip to combined_lora_adapter.zip


In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [ ]:
import json, zipfile, os, torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel


ADAPTER_ZIP  = "./combined_lora_adapter.zip"
ADAPTER_DIR  = "./combined_lora_adapter"
MODEL_NAME   = "google/flan-t5-xl"
MAX_INPUT    = 128
MAX_TARGET   = 384


if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted adapter to:", ADAPTER_DIR)


dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)
print(f"Loading {MODEL_NAME} ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto"
)
print("Loading LoRA weights...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = True

try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


Extracted adapter to: ./combined_lora_adapter
Loading google/flan-t5-xl (torch.float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading LoRA weights...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [ ]:

def generate_plan(question, dataset="Wikidata"):

    prompt = f"decompose question to steps [{dataset}]: {question}"
    inp = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT,
        truncation=True,
    )
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=MAX_TARGET,
            num_beams=4,
            early_stopping=True,
            length_penalty=1.0,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def predict(question, dataset="Wikidata"):
    """
    dataset: 'Wikidata' or 'DBpedia'
    """
    plan = generate_plan(question, dataset)


    print(f"  Question : {question}")
    print(f"  Dataset  : {dataset}")

    print(f"\n   Generated Plan:")

    for step in plan.split('||'):
        print(f"  {step.strip()}")

    return plan



In [ ]:
question = "Which was the wife of Erich Honecker in the series ordinal 3?"
dataset  = "Wikidata"    # "Wikidata" or "DBpedia"

plan = predict(question, dataset)

  Question : Which was the wife of Erich Honecker in the series ordinal 3?
  Dataset  : Wikidata

   Generated Plan:
  step1: action: find_entity | surface_form: Erich Honecker | output_variable:?erichhonecker | semantic_type: ENTITY
  step2: action: find_object | property: spouse | subject_variable:?erichhonecker | output_variable:?answer | semantic_type: PROPERTY
  step3: action: find_object | property: series ordinal | subject_variable:?answer | output_variable:?x | semantic_type: PROPERTY
  step4: action: filter | filter_variable:?x | operator: contains | value: 3 | value_type: string | semantic_type: LITERAL


In [ ]:
plan

'step1: action: find_entity | surface_form: Erich Honecker | output_variable:?erichhonecker | semantic_type: ENTITY || step2: action: find_object | property: spouse | subject_variable:?erichhonecker | output_variable:?answer | semantic_type: PROPERTY || step3: action: find_object | property: series ordinal | subject_variable:?answer | output_variable:?x | semantic_type: PROPERTY || step4: action: filter | filter_variable:?x | operator: contains | value: 3 | value_type: string | semantic_type: LITERAL'

In [ ]:
question

'Which was the wife of Erich Honecker in the series ordinal 3?'